# RevuLens Pilot Experiment (Colab-Ready Notebook)
## Day 1: Data Preprocessing, Stratified Split & TF-IDF Baseline

Thesis Pilot comparing frozen encoders against TF-IDF SVM baseline for Genuine (0) vs Computer-Generated (1) review classification.

In [ ]:
# --- Configuration & Paths ---
# Change these paths if running in Google Colab (e.g. '/content/fake reviews dataset.csv')
SALMINEN_DATA_PATH = r'fake reviews dataset.csv'  # update to local or colab path
RANDOM_SEED = 42
RESULTS_DIR = 'results'
DATA_CACHE_DIR = 'data'

import os
import re
import html
import json
import random
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    brier_score_loss, confusion_matrix
)

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed(RANDOM_SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_CACHE_DIR, exist_ok=True)
print('Setup complete. Seed set to', RANDOM_SEED)

In [ ]:
# --- Text Normalization ---
def normalize_text(text: Any) -> str:
    if not isinstance(text, str):
        return ''
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)  # remove HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # remove URLs
    text = text.lower()  # lowercase
    text = re.sub(r'([!?,.:;])\1+', r'\1', text)  # normalize repeated punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # normalize whitespace
    return text

print('Normalized sample:', normalize_text('<div>Check https://example.com!!! BEST product ever!</div>'))

In [ ]:
# --- Dataset Loader (Supports Salminen and Future Schema) ---
def load_dataset(filepath: str, verbose: bool = True) -> pd.DataFrame:
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'File not found: {filepath}')
    df = pd.read_csv(filepath)
    
    text_col = 'text_' if 'text_' in df.columns else ('text' if 'text' in df.columns else 'review')
    df['clean_text'] = df[text_col].apply(normalize_text)
    
    label_map = {
        'OR': 0, 'or': 0, 'Original': 0, 'genuine': 0, 'Genuine': 0, 0: 0, 0.0: 0, '0': 0,
        'CG': 1, 'cg': 1, 'Computer-Generated': 1, 'fake': 1, 'Fake': 1, 1: 1, 1.0: 1, '1': 1
    }
    df['label'] = df['label'].map(label_map).astype(int)
    if 'language' not in df.columns:
        df['language'] = 'English'
        
    # Drop empty and duplicates
    df = df[df['clean_text'].str.strip() != '']
    df = df.drop_duplicates(subset=['clean_text'], keep='first')
    df['text'] = df['clean_text']
    
    if verbose:
        print(f'Loaded {len(df)} valid records. Labels:\n{df["label"].value_counts()}')
    return df[['text', 'label', 'language']].reset_index(drop=True)

In [ ]:
# --- Stratified 80/10/10 Split ---
def split_dataset(df: pd.DataFrame, seed: int = RANDOM_SEED, save_path: Optional[str] = None):
    if save_path and os.path.exists(save_path):
        with open(save_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    indices = np.arange(len(df))
    labels = df['label'].values
    train_idx, temp_idx = train_test_split(indices, test_size=0.20, random_state=seed, stratify=labels)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=seed, stratify=labels[temp_idx])
    splits = {'train': train_idx.tolist(), 'val': val_idx.tolist(), 'test': test_idx.tolist()}
    if save_path:
        os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(splits, f, indent=2)
    return splits

In [ ]:
# --- Calibration & Evaluation Metrics ---
class ECEEvaluator:
    @staticmethod
    def compute(y_true: np.ndarray, prob_pos: np.ndarray, n_bins: int = 10) -> float:
        bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
        ece = 0.0
        n = len(y_true)
        for i in range(n_bins):
            b_low, b_high = bin_boundaries[i], bin_boundaries[i+1]
            in_bin = (prob_pos >= b_low) & (prob_pos <= b_high if i == n_bins - 1 else prob_pos < b_high)
            count = np.sum(in_bin)
            if count > 0:
                acc_bin = np.mean(y_true[in_bin])
                conf_bin = np.mean(prob_pos[in_bin])
                ece += (count / n) * np.abs(acc_bin - conf_bin)
        return float(ece)

def compute_metrics(y_true: np.ndarray, prob_cg: np.ndarray, threshold: float = 0.50):
    y_pred = (prob_cg >= threshold).astype(int)
    confidences = np.maximum(prob_cg, 1.0 - prob_cg)
    acc = accuracy_score(y_true, y_pred)
    prec_cg = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec_cg = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1_cg = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    brier = brier_score_loss(y_true, prob_cg)
    ece = ECEEvaluator.compute(y_true, prob_cg, n_bins=10)
    corr_mask = (y_pred == y_true)
    incorr_mask = (y_pred != y_true)
    return {
        'accuracy': float(acc),
        'precision_cg': float(prec_cg),
        'recall_cg': float(rec_cg),
        'f1_cg': float(f1_cg),
        'brier_score': float(brier),
        'ece': float(ece),
        'mean_conf_correct': float(np.mean(confidences[corr_mask])) if np.any(corr_mask) else 0.0,
        'mean_conf_incorrect': float(np.mean(confidences[incorr_mask])) if np.any(incorr_mask) else 0.0,
        'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
    }

In [ ]:
# --- TF-IDF Baseline Training & Calibration ---
def train_baseline_tfidf(train_texts, train_labels, val_texts, val_labels, c_grid=None):
    if c_grid is None:
        c_grid = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
    vec = TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True)
    X_tr = vec.fit_transform(train_texts)
    X_va = vec.transform(val_texts)
    
    scaler = StandardScaler(with_mean=False)
    X_tr_sc = scaler.fit_transform(X_tr)
    X_va_sc = scaler.transform(X_va)
    
    best_c, best_f1 = c_grid[0], -1.0
    for c in c_grid:
        svc = LinearSVC(C=c, random_state=RANDOM_SEED, max_iter=2000, dual=False)
        svc.fit(X_tr_sc, train_labels)
        f1 = f1_score(val_labels, svc.predict(X_va_sc), pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_c = f1, c
            
    print(f'Selected optimal C: {best_c} (Validation F1={best_f1:.4f})')
    base_svc = LinearSVC(C=best_c, random_state=RANDOM_SEED, max_iter=2000, dual=False)
    calibrated = CalibratedClassifierCV(estimator=base_svc, method='sigmoid', cv=5)
    calibrated.fit(X_tr_sc, train_labels)
    val_probas = calibrated.predict_proba(X_va_sc)[:, 1]
    metrics = compute_metrics(val_labels, val_probas)
    return calibrated, scaler, vec, metrics, val_probas